# 01 - First look at MovieLens 32M

Goal: look at the raw files (read-only), no cleaning yet.

In [1]:
import duckdb

con = duckdb.connect()

## Structure of the files (first rows)

In [2]:
# First 5 ratings
con.sql("SELECT * FROM '../data/raw/ml-32m/ratings.csv' LIMIT 5")

┌────────┬─────────┬────────┬───────────┐
│ userId │ movieId │ rating │ timestamp │
│ int64  │  int64  │ double │   int64   │
├────────┼─────────┼────────┼───────────┤
│      1 │      17 │    4.0 │ 944249077 │
│      1 │      25 │    1.0 │ 944250228 │
│      1 │      29 │    2.0 │ 943230976 │
│      1 │      30 │    5.0 │ 944249077 │
│      1 │      32 │    5.0 │ 943228858 │
└────────┴─────────┴────────┴───────────┘

In [3]:
# First 5 movies
con.sql("SELECT * FROM '../data/raw/ml-32m/movies.csv' LIMIT 5")

┌─────────┬────────────────────────────────────┬─────────────────────────────────────────────┐
│ movieId │               title                │                   genres                    │
│  int64  │              varchar               │                   varchar                   │
├─────────┼────────────────────────────────────┼─────────────────────────────────────────────┤
│       1 │ Toy Story (1995)                   │ Adventure|Animation|Children|Comedy|Fantasy │
│       2 │ Jumanji (1995)                     │ Adventure|Children|Fantasy                  │
│       3 │ Grumpier Old Men (1995)            │ Comedy|Romance                              │
│       4 │ Waiting to Exhale (1995)           │ Comedy|Drama|Romance                        │
│       5 │ Father of the Bride Part II (1995) │ Comedy                                      │
└─────────┴────────────────────────────────────┴─────────────────────────────────────────────┘

## Size of `ratings.csv`: ratings, users, movies, period, scale

In [ ]:
# number of ratings 
con.sql("SELECT COUNT(*) FROM '../data/raw/ml-32m/ratings.csv' ")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     32000204 │
└──────────────┘

In [9]:
# how many different users ? 
con.sql("SELECT COUNT(DISTINCT userId) FROM '../data/raw/ml-32m/ratings.csv' ")

┌────────────────────────┐
│ count(DISTINCT userId) │
│         int64          │
├────────────────────────┤
│                 200948 │
└────────────────────────┘

In [11]:
# how many distinct movies have been rated (so how many distinct movies in the DB Ratings)
con.sql("SELECT COUNT(DISTINCT movieId) FROM '../data/raw/ml-32m/ratings.csv' ")

┌─────────────────────────┐
│ count(DISTINCT movieId) │
│          int64          │
├─────────────────────────┤
│                   84432 │
└─────────────────────────┘

In [15]:
con.sql("SELECT to_timestamp(MIN(timestamp)), to_timestamp(MAX(timestamp)) FROM '../data/raw/ml-32m/ratings.csv'")

┌────────────────────────────────┬────────────────────────────────┐
│ to_timestamp(min("timestamp")) │ to_timestamp(max("timestamp")) │
│    timestamp with time zone    │    timestamp with time zone    │
├────────────────────────────────┼────────────────────────────────┤
│ 1995-01-09 12:46:44+01         │ 2023-10-13 04:29:07+02         │
└────────────────────────────────┴────────────────────────────────┘

In [ ]:
# scale of ratings
con.sql("SELECT DISTINCT rating FROM '../data/raw/ml-32m/ratings.csv' ORDER BY rating ASC")

┌────────┐
│ rating │
│ double │
├────────┤
│    0.5 │
│    1.0 │
│    1.5 │
│    2.0 │
│    2.5 │
│    3.0 │
│    3.5 │
│    4.0 │
│    4.5 │
│    5.0 │
└────────┘

## How MovieLens writes titles (articles, years, alternative titles)

In [18]:
con.sql("SELECT movieId, title FROM '../data/raw/ml-32m/movies.csv' WHERE title LIKE '%Godfather%' OR title LIKE '%Titanic%' OR title LIKE '%Amelie%'")

┌─────────┬──────────────────────────────────────────────────────────────────────────┐
│ movieId │                                  title                                   │
│  int64  │                                 varchar                                  │
├─────────┼──────────────────────────────────────────────────────────────────────────┤
│     858 │ Godfather, The (1972)                                                    │
│    1221 │ Godfather: Part II, The (1974)                                           │
│    1721 │ Titanic (1997)                                                           │
│    2023 │ Godfather: Part III, The (1990)                                          │
│    2157 │ Chambermaid on the Titanic, The (Femme de chambre du Titanic, La) (1998) │
│    3403 │ Raise the Titanic (1980)                                                 │
│    3404 │ Titanic (1953)                                                           │
│    4864 │ Titanica (1992)                

## What we learned
- 32,000,204 ratings, 200,948 users, 84,432 rated movies.
- Ratings go from 1995-01-09 to 2023-10-13 (2023 is incomplete), on a 0.5 to 5.0 scale in steps of 0.5.
- The release year is inside the title, e.g. `Toy Story (1995)`, and genres are separated by `|`.
- Next: `02_explore_oscars.ipynb`.